In [1]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.utils import shuffle

# 가상의 데이터셋 생성 (CIFAR-10과 유사한 데이터셋)
num_classes = 10
input_shape = (32, 32, 3)
num_samples = 1000

X = np.random.rand(num_samples, *input_shape)  # 1000개의 32x32x3 이미지 생성
y = np.random.randint(num_classes, size=num_samples)  # 0부터 9까지의 정수 레이블 생성

# 데이터셋 셔플 및 분할
X, y = shuffle(X, y)  # 데이터를 섞어줌
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)  # 80% 훈련, 20% 테스트

In [2]:
from tensorflow.keras import backend as K
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D
from tensorflow.keras.optimizers import Adam

# === 각 하이퍼파라미터 trial 시작 시 가장 먼저 호출 ===
K.clear_session()

def create_model(learning_rate=0.001):
    base_model = ResNet50(weights='imagenet', include_top=False, input_shape=input_shape)
    base_model.trainable = False          # 전이학습 안정화 (가장 중요)

    x = base_model.output
    x = GlobalAveragePooling2D()(x)  # 글로벌 평균 풀링 레이어 추가
    x = Dense(1024, activation='relu')(x)  # 완전 연결 레이어 추가
    predictions = Dense(num_classes, activation='softmax')(x)  # 출력 레이어 추가

    model = Model(inputs=base_model.input, outputs=predictions)
    opt = Adam(learning_rate=learning_rate, clipnorm=1.0)  # Adam 옵티마이저 설정, gradient clipping 1.0

    model.compile(optimizer=opt, loss='sparse_categorical_crossentropy', metrics=['accuracy'])  # 모델 컴파일
    return model

In [3]:
learning_rates = [0.001, 0.01, 0.1]
batch_sizes = [16, 32, 64]
epochs = 10

best_accuracy = 0
best_params = {}

# 하이퍼파라미터 조합을 수동으로 설정하여 모델 학습 및 평가
for lr in learning_rates:
    for batch_size in batch_sizes:
        model = create_model(learning_rate=lr)
        model.fit(X_train, y_train, epochs=epochs, batch_size=batch_size, validation_split=0.2, verbose=1)

        loss, accuracy = model.evaluate(X_test, y_test, verbose=0)
        print(f"Learning Rate: {lr}, Batch Size: {batch_size}, Test Accuracy: {accuracy}")

        if accuracy > best_accuracy:
            best_accuracy = accuracy
            best_params = {'learning_rate': lr, 'batch_size': batch_size}

print("Best Parameters:", best_params)
print("Best Test Accuracy:", best_accuracy)

94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 6s 0us/step
Epoch 1/10
40/40 ━━━━━━━━━━━━━━━━━━━━ 17s 122ms/step - accuracy: 0.1141 - loss: 3.3666 - val_accuracy: 0.1375 - val_loss: 3.3538
Epoch 2/10
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.0891 - loss: 2.7672 - val_accuracy: 0.0750 - val_loss: 2.4692
Epoch 3/10
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.0812 - loss: 2.5693 - val_accuracy: 0.0688 - val_loss: 2.7006
Epoch 4/10
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.0938 - loss: 2.4196 - val_accuracy: 0.0750 - val_loss: 2.3221
Epoch 5/10
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.0875 - loss: 2.3123 - val_accuracy: 0.0750 - val_loss: 2.3036
Epoch 6/10
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.1109 - loss: 2.3061 - val_accuracy: 0.0750 - val_loss: 2.3050
Epoch 7/10
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.0766 - loss: 2.3034 - val_accuracy: 0.1125 - val_loss: 2.3023
Epoch 8/10
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.

In [4]:
# 최적의 하이퍼파라미터로 모델 재학습
epochs = 10
best_model = create_model(learning_rate=best_params['learning_rate'])
best_model.fit(X_train, y_train, epochs=epochs, batch_size=best_params['batch_size'], validation_split=0.2)

# 테스트 데이터셋으로 모델 평가
test_loss, test_accuracy = best_model.evaluate(X_test, y_test)
print(f"Test Accuracy: {test_accuracy:.2f}")
print(f"Test Loss: {test_loss:.2f}")

Epoch 1/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 19s 791ms/step - accuracy: 0.1047 - loss: 26.6700 - val_accuracy: 0.0750 - val_loss: 2.3206
Epoch 2/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.1000 - loss: 2.3140 - val_accuracy: 0.0750 - val_loss: 2.3028
Epoch 3/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.1109 - loss: 2.3026 - val_accuracy: 0.0750 - val_loss: 2.3030
Epoch 4/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.1109 - loss: 2.3014 - val_accuracy: 0.0750 - val_loss: 2.3027
Epoch 5/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.1109 - loss: 2.3001 - val_accuracy: 0.0750 - val_loss: 2.3028
Epoch 6/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.1109 - loss: 2.2991 - val_accuracy: 0.0750 - val_loss: 2.3025
Epoch 7/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step - accuracy: 0.1109 - loss: 2.2980 - val_accuracy: 0.0750 - val_loss: 2.3022
Epoch 8/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.1109 - loss: 2.2972 - val_accuracy: 0.0750 